In [1]:
import math
from typing import Callable
import numpy as np

def est_D(probabilidades: list[float]):
    n = len(probabilidades)
    prob = sorted(probabilidades)

    D_p = [((i + 1)/n - prob[i]) for i in range(n)]  # Fe(Y) - F(Y)
    D_m = [(prob[i] - i/n) for i in range(n)]  # F(Y) - Fe(Y)

    d = max(max(D_p), max(D_m))

    return d


def estimar_p_valor_simulaciones_continua(
    d_obs: float,
    gen_prob: Callable[[], list[float]],
    N_SIM: int = 10_000
):
    # Se utiliza el estadístico de Kolmogorov-Smirnov

    p_valor = 0.0
    for _ in range(N_SIM):
        probabilidades = gen_prob()
        d_j = est_D(probabilidades)

        if d_j >= d_obs:
            p_valor += 1

    p_valor = p_valor / N_SIM
    return p_valor

### Ejercicio 9.
En un estudio de vibraciones, una muestra aleatoria de $15$ componentes del avión fueron sometidos a fuertes vibraciones hasta que se evidenciaron fallas estructurales. Los datos proporcionados son los minutos transcurridos hasta que se evidenciaron dichas fallas.
$$ 1.6, 10.3, 3.5, 13.5, 18.4, 7.7, 24.3, 10.7, 8.4, 4.9, 7.9, 12, 16.2, 6.8, 14.7 $$

Pruebe la hipótesis nula de que estas observaciones pueden ser consideradas como una muestra de la distribución exponencial.

---

### Análisis
$$
\begin{array}{cl}
    H_0 & = \text{"El tiempo de falla de los componentes de un avion tiene distribución exponencial"} \\
    H_1 & = \text{"El tiempo de falla de los componentes de un avion no tiene distribución exponencial"}
\end{array}
$$

Como se desconoce $\lambda$. Entonces utilizare que $\lambda = \frac{1}{\mu}$ y estimare $\hat{\lambda} = \frac{1}{\overline{X}(15)}$

Asumo $\alpha = 0.05$

In [2]:
n = 15
datos = [1.6, 10.3, 3.5, 13.5, 18.4, 7.7, 24.3, 10.7, 8.4, 4.9, 7.9, 12, 16.2, 6.8, 14.7]

assert len(datos) == n

# estimo lambda
lamda = 1 / np.mean(datos)

# Calculando las prob utilizando la acumulada
probabilidades = [(1 - math.exp(-(datos[i]) * lamda)) for i in range(n)]
d_obs = est_D(probabilidades)

def gen_prob():
    # generar n muestras con la distribución F_H0
    muestra_j = np.random.exponential(1/lamda, n)

    # estimo lambda
    l_j = 1 / np.mean(muestra_j)

    # Calculando las prob utilizando la acumulada
    prob = [(1 - math.exp(-(muestra_j[i]) * l_j)) for i in range(n)]
    return prob

p_valor = estimar_p_valor_simulaciones_continua(
    d_obs,
    gen_prob
)

print("d_obs: ", d_obs)
print("p_valor: ", p_valor)

d_obs:  0.26949936321059237
p_valor:  0.0469


Como $\text{p-valor} \le \alpha \equiv 0.0469 \le 0.05$, entonces se rechaza $H_0$. 